# Stage 4 — Noise Topic Removal

Removes topics that are clearly non-NS content (Reddit bot messages, UI noise,
moderation messages, emoji/emote clusters, Reddit meta-discussion).

Approach:
- Set noise topic assignments to -1 in the topics array
- Call `update_topics()` to recalculate c-TF-IDF representations without noise chunks
- Save updated model, re-export keywords, update all chunk parquets

Topic merging is intentionally skipped — hierarchical dendrogram structure must
be preserved for downstream macro labelling.

**Datasets needed:**
- `ns-sentiment-chunks-v3` — `submissions_chunks.parquet`, `comments_chunks.parquet`
- `ns-sentiment-bertopic-v3` — `bertopic_fine/` model directory, `chunk_topics.parquet`

**Setup:** GPU T4 x2 · Save & Run All

In [ ]:
# Cell 1 — Discover input paths
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Cell 2 — Config  <- UPDATE PATHS AFTER RUNNING CELL 1
SUBMISSIONS_CHUNKS = '/kaggle/input/<your-dataset>/submissions_chunks.parquet'
COMMENTS_CHUNKS    = '/kaggle/input/<your-dataset>/comments_chunks.parquet'
CHUNK_TOPICS       = '/kaggle/input/<your-dataset>/chunk_topics.parquet'
BERTOPIC_FINE_PATH = '/kaggle/input/<your-dataset>/bertopic_fine'
OUT_DIR            = '/kaggle/working'

# Noise topics confirmed for removal
# t58/t127: AutoMod bot messages
# t77/t128/t255: Reddit UI noise (menu, clicking, submitting)
# t79/t207: Reddit moderation (moderator, removed)
# t94: Reddit megathread/mods noise
# t257/t261: Reddit meta rules (asksingapore, sgexams, trivial)
# t372: Reddit emoji/emote noise
# t89/t195: Subreddit rules boilerplate (identical AutoMod acknowledgement posts)
# t97: Deleted user content + technical junk strings
# t71/t117: Generic filler/outlier content (added Stage 4 label vetting)
NOISE_TOPIC_IDS = [58, 71, 77, 79, 89, 94, 97, 117, 127, 128, 195, 207, 255, 257, 261, 372]

In [ ]:
# Cell 3 — Imports
!pip install bertopic safetensors sentence-transformers -q

import numpy as np
import pandas as pd
from bertopic import BERTopic

print('Imports OK')

In [ ]:
# Cell 4 — Load model and data
print('Loading BERTopic model ...')
topic_model = BERTopic.load(BERTOPIC_FINE_PATH)
print(f'Model loaded — {len(topic_model.get_topic_info()) - 1} topics')

print('Loading chunk parquets ...')
sub = pd.read_parquet(SUBMISSIONS_CHUNKS)
com = pd.read_parquet(COMMENTS_CHUNKS)
df  = pd.concat([sub, com], ignore_index=True)
del sub, com
print(f'Total chunks: {len(df):,}')

docs = df['text'].tolist()

print('Loading chunk_topics ...')
ct = pd.read_parquet(CHUNK_TOPICS)
# Drop duplicates before merging (minor batch boundary issue in source parquet)
ct = ct.drop_duplicates(subset='chunk_id', keep='first')
# Align to df order via left merge on chunk_id
ct = df[['chunk_id']].merge(ct, on='chunk_id', how='left')
topics_current = ct['topic_id_fine'].to_numpy(dtype=np.int16)
print(f'Topics loaded — current outliers: {(topics_current == -1).sum():,}')

In [ ]:
# Cell 5 — Apply noise removal
noise_chunks = np.isin(topics_current, NOISE_TOPIC_IDS).sum()
print(f'Chunks in noise topics: {noise_chunks:,}')
print(f'Outliers before:        {(topics_current == -1).sum():,}')

# Set noise topic assignments to -1
topics_cleaned = topics_current.copy()
topics_cleaned[np.isin(topics_cleaned, NOISE_TOPIC_IDS)] = -1

print(f'Outliers after:         {(topics_cleaned == -1).sum():,}')
print(f'Topics remaining:       {len(set(topics_cleaned)) - 1}')

# Recalculate c-TF-IDF for all remaining topics, excluding noise chunks entirely
print('Recalculating topic representations via update_topics() ...')
topic_model.update_topics(docs, topics=topics_cleaned.tolist())
print('Done')

In [ ]:
# Cell 6 — Save updated model and re-export keywords
topic_model.save(
    f'{OUT_DIR}/bertopic_fine',
    serialization='safetensors',
    save_ctfidf=True,
)
print('Saved bertopic_fine')

rows = []
for _, row in topic_model.get_topic_info().iterrows():
    tid = row['Topic']
    if tid == -1:
        continue
    kws = topic_model.get_topic(tid)
    rows.append({
        'topic_id': tid,
        'count':    row['Count'],
        'name':     row.get('Name', ''),
        'keywords': ', '.join(w for w, _ in kws[:10]),
        'scores':   ', '.join(f'{s:.4f}' for _, s in kws[:10]),
    })
kw_df = pd.DataFrame(rows)
kw_df.to_csv(f'{OUT_DIR}/topic_keywords_fine.csv', index=False)
print(f'Saved topic_keywords_fine.csv  ({len(kw_df)} topics)')

In [ ]:
# Cell 7 — Update chunk_topics and write back to chunk parquets
ct_out = ct.copy()
ct_out['topic_id_fine'] = topics_cleaned

# Clear coarse for chunks that were noise (now outliers)
was_noise = np.isin(topics_current, NOISE_TOPIC_IDS)
ct_out.loc[was_noise, 'topic_id_coarse'] = -1

ct_out.to_parquet(f'{OUT_DIR}/chunk_topics.parquet', index=False)
print(f'Saved chunk_topics.parquet  ({len(ct_out):,} rows)')

merge_cols = ['chunk_id', 'topic_id_fine', 'topic_prob_fine', 'topic_id_coarse']
for fname, src in [('submissions_chunks.parquet', SUBMISSIONS_CHUNKS),
                   ('comments_chunks.parquet',    COMMENTS_CHUNKS)]:
    part = pd.read_parquet(src)
    part = part.drop(columns=[c for c in merge_cols[1:] if c in part.columns])
    part = part.merge(ct_out[merge_cols], on='chunk_id', how='left')
    part.to_parquet(f'{OUT_DIR}/{fname}', index=False)
    print(f'Saved {fname}  ({len(part):,} rows)')

In [ ]:
# Cell 8 — Quality summary
outlier_rate = (topics_cleaned == -1).mean() * 100
n_topics     = len(set(topics_cleaned)) - 1

print('── Noise removal summary ──')
print(f'Noise topics removed:   {len(NOISE_TOPIC_IDS)}')
print(f'Chunks -> outliers:     {noise_chunks:,}')
print(f'Remaining fine topics:  {n_topics}')
print(f'Outlier rate:           {outlier_rate:.2f}%')
print()
print('── Top 10 topics by size ──')
print(kw_df.nlargest(10, 'count')[['topic_id', 'count', 'keywords']].to_string(index=False))